In [2]:
import tensorflow as tf

In [15]:
def positional_encoding(length,dims):
    pos=tf.cast(tf.range(length),dtype=tf.float32)[:,tf.newaxis]
    dim_d=tf.cast(tf.range(dims),dtype=tf.float32)[tf.newaxis,:]
    angels=tf.cast(pos,dtype=tf.float32)/tf.pow(10000,2*(dim_d//2)/tf.cast(dims,tf.float32))
    pos_enc=tf.where(dim_d%2==0,tf.sin(angels),tf.cos(angels))
    return pos_enc

In [16]:
pos_enc = positional_encoding(128, 64)

print("Shape:", pos_enc.shape)
print("First position:")
print(pos_enc[0, :10].numpy())

print("\nSecond position:")
print(pos_enc[1, :10].numpy())

print("\nContains NaN:", tf.reduce_any(tf.math.is_nan(pos_enc)).numpy())
print("Contains Inf:", tf.reduce_any(tf.math.is_inf(pos_enc)).numpy())

Shape: (128, 64)
First position:
[0. 1. 0. 1. 0. 1. 0. 1. 0. 1.]

Second position:
[0.841471   0.5403023  0.68156135 0.731761   0.53316844 0.84600914
 0.40930894 0.91239583 0.3109836  0.95041525]

Contains NaN: False
Contains Inf: False


In [17]:
def FullyConnectedLayer(embedding_dims,ff_dims):
    return tf.keras.Sequential([
        tf.keras.layers.Dense(ff_dims,activation='relu'),
        tf.keras.layers.Dense(embedding_dims)
    ])

In [19]:
class Encoder_Layer(tf.keras.layers.Layer):

    def __init__(
        self,
        embedding_dims,
        ff_dims,
        num_heads,
        dropout_rate=0.2,
        layernorm_eps=1e-6
    ):
        super().__init__()

        self.mha = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embedding_dims // num_heads,
            dropout=dropout_rate
        )

        self.ffn = FullyConnectedLayer(embedding_dims, ff_dims)

        self.layer_norm1 = tf.keras.layers.LayerNormalization(
            epsilon=layernorm_eps
        )
        self.layer_norm2 = tf.keras.layers.LayerNormalization(
            epsilon=layernorm_eps
        )

        self.dropout = tf.keras.layers.Dropout(dropout_rate)

    def call(self, x, mask=None, training=None):

        attention_output = self.mha(
            query=x,
            key=x,
            value=x,
            attention_mask=mask,
            training=training
        )

        attention_output = self.layer_norm1(
            x + attention_output
        )

        ffn_output = self.ffn(attention_output)

        ffn_output = self.dropout(
            ffn_output,
            training=training
        )

        output = self.layer_norm2(
            attention_output + ffn_output
        )

        return output

In [20]:
class Encoder(tf.keras.layers.Layer):

    def __init__(
        self,
        num_layers,
        input_vacab_size,
        embedding_dims,
        maximum_position_encoding,
        ff_dims,
        num_heads,
        dropout_rate=0.2,
        layernorm_eps=1e-6
    ):
        super().__init__()

        self.embedding = tf.keras.layers.Embedding(
            input_vacab_size,
            embedding_dims
        )

        self.positional_encoding = positional_encoding(
            maximum_position_encoding,
            embedding_dims
        )

        self.enc_layers = [
            Encoder_Layer(
                embedding_dims,
                ff_dims,
                num_heads,
                dropout_rate,
                layernorm_eps
            )
            for _ in range(num_layers)
        ]

        self.num_layers = num_layers
        self.dropout = tf.keras.layers.Dropout(dropout_rate)
        self.embedding_dims = embedding_dims

    def call(self, x, training=None, mask=None):

        seq_len = tf.shape(x)[1]

        x = self.embedding(x)

        x *= tf.math.sqrt(
            tf.cast(self.embedding_dims, tf.float32)
        )

        x += self.positional_encoding[:seq_len, :]

        x = self.dropout(x, training=training)

        for layer in self.enc_layers:
            x = layer(
                x,
                mask=mask,
                training=training
            )

        return x

In [21]:
class Decoder_Layer(tf.keras.layers.Layer):
    def __init__(self,embedding_dims,num_heads,ff_dims,dropout_rate=0.2,layernorm_eps=1e-6):
        super().__init__()
        self.mha1=tf.keras.layers.MultiHeadAttention(num_heads=num_heads,key_dim=embedding_dims// num_heads,dropout=dropout_rate)
        self.mha2=tf.keras.layers.MultiHeadAttention(num_heads=num_heads,key_dim=embedding_dims// num_heads,dropout=dropout_rate)
        self.ffn=FullyConnectedLayer(embedding_dims,ff_dims)
        self.layernorm1=tf.keras.layers.LayerNormalization(epsilon=layernorm_eps)
        self.layernorm2=tf.keras.layers.LayerNormalization(epsilon=layernorm_eps)
        self.layernorm3=tf.keras.layers.LayerNormalization(epsilon=layernorm_eps)
        self.dropout=tf.keras.layers.Dropout(rate=dropout_rate)
    def call(self,x,enc_out,look_ahead_mask=None,padding_mask=None,training=None):
        mh1 = self.mha1(query=x, key=x, value=x,   attention_mask=look_ahead_mask,  training=training)
        nor=self.layernorm1(x+mh1)
        mh2=self.mha2(nor,enc_out,enc_out,attention_mask=padding_mask,training=training)
        nor2=self.layernorm2(nor+mh2)
        ffn_out=self.ffn(nor2)
        dropout=self.dropout(ffn_out,training=training)
        norm3=self.layernorm3(dropout+nor2)
        return norm3




In [22]:
class Decoder(tf.keras.layers.Layer):
    def __init__(self,num_layers,vocab_size,embedding_dims,maximum_position_encoding,num_heads,ff_dims,dropout_rate=0.2,layernorm_eps=1e-6):
        super().__init__()
        self.num_layers=num_layers
        self.embedding=tf.keras.layers.Embedding(vocab_size,embedding_dims)
        self.encoding=positional_encoding(maximum_position_encoding,embedding_dims)
        self.dec_layers=[Decoder_Layer(embedding_dims,num_heads,ff_dims,dropout_rate,layernorm_eps)
                        for _ in range(num_layers)]
        self.dropout = tf.keras.layers.Dropout(rate=dropout_rate)
        self.embedding_dims=embedding_dims
    def call(self,x,enc_output,training=None,padding_mask=None,look_ahead_mask=None):
        seq_length=tf.shape(x)[1]
        x=self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.embedding_dims, tf.float32))
        x += self.encoding[:seq_length, :]
        x = self.dropout(x, training=training)
        for i in range(self.num_layers):
            x=self.dec_layers[i](x,enc_output,look_ahead_mask=look_ahead_mask,padding_mask=padding_mask,training=training)
        return x



In [23]:
class Transformer(tf.keras.Model):
    def __init__(self,num_layers,input_vocab_size,embedding_dims,maximum_position_encoding,ff_dims,num_heads,target_vocab_size):
        super().__init__()
        self.encoder=Encoder(num_layers,input_vocab_size,embedding_dims,maximum_position_encoding,ff_dims,num_heads)
        self.decoder = Decoder(
    num_layers,
    target_vocab_size,
    embedding_dims,
    maximum_position_encoding,
    num_heads,     # Correct position
    ff_dims        # Correct position
)
        self.final_layer = tf.keras.layers.Dense(target_vocab_size)

    def call(self, input_sentence, output_sentence, enc_padding_mask, look_ahead_mask, dec_padding_mask,training=None):
        enc_output = self.encoder( input_sentence,training=training,mask=enc_padding_mask)
        decoder_output=self.decoder(output_sentence,enc_output,training=training,padding_mask=dec_padding_mask,look_ahead_mask=look_ahead_mask)
        final_output = self.final_layer(decoder_output)
        return final_output







In [24]:
# Example configuration
input_vocab_size = 100
target_vocab_size = 120

transformer = Transformer(
    num_layers=2,
    input_vocab_size=input_vocab_size,
    embedding_dims=16,
    maximum_position_encoding=50,
    ff_dims=64,
    num_heads=4,
    target_vocab_size=target_vocab_size
)

# Two source sentences; 0 represents padding
input_sentence = tf.constant([
    [4, 7, 9, 0],
    [3, 8, 0, 0]
])

# Decoder input sentences
output_sentence = tf.constant([
    [1, 5, 6],
    [1, 9, 0]
])

# Create masks
enc_padding_mask = tf.not_equal(
    input_sentence[:, tf.newaxis, :], 0
)

dec_padding_mask = tf.not_equal(
    input_sentence[:, tf.newaxis, :], 0
)

target_padding_mask = tf.not_equal(
    output_sentence[:, tf.newaxis, :], 0
)

seq_len = tf.shape(output_sentence)[1]

# True for allowed positions; False for future positions
look_ahead_mask = tf.linalg.band_part(
    tf.ones((seq_len, seq_len), dtype=tf.bool),
    -1, 0
)

# Combine target padding and causal masks
look_ahead_mask = tf.logical_and(
    target_padding_mask,
    look_ahead_mask[tf.newaxis, :, :]
)

# Run the model
predictions = transformer(
    input_sentence,
    output_sentence,

    enc_padding_mask=enc_padding_mask,
    look_ahead_mask=look_ahead_mask,
    dec_padding_mask=enc_padding_mask,
    training=False
)

print("Predictions shape:", predictions.shape)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder__layer' (of type Encoder_Layer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder__layer_1' (of type Encoder_Layer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder' (of type Encoder) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Predictions shape: (2, 3, 120)


In [25]:
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True,
    reduction="none"
)
def masked_loss(real,prediction):
    loss=loss_object(real,prediction)
    mask=tf.cast( tf.not_equal(real, 0),dtype=loss.dtype)
    loss *= mask
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

In [26]:
def create_padding_mask(token_ids):
    return tf.math.not_equal(token_ids, 0)[:,tf.newaxis,:]
def create_lookahead_mask(seq_length):
    mask=tf.linalg.band_part(tf.ones((1, seq_length, seq_length),dtype=tf.bool),-1,0)
    return mask

In [27]:
@tf.function
def train_step(model, input_data, target_data, optimizer):

    dec_input = target_data[:, :-1]
    dec_real = target_data[:, 1:]

    enc_mask = create_padding_mask(input_data)
    target_mask = create_padding_mask(dec_input)
    look_mask = create_lookahead_mask(tf.shape(dec_input)[1])

    combined_mask = tf.logical_and(target_mask, look_mask)

    with tf.GradientTape() as tape:
        output = model(
            input_data,
            dec_input,
            enc_mask,
            combined_mask,
            enc_mask,
            training=True
        )

        loss = masked_loss(dec_real, output)

    gradients = tape.gradient(loss, model.trainable_variables)

    optimizer.apply_gradients(
        zip(gradients, model.trainable_variables)
    )

    return loss

In [28]:
import tensorflow as tf
import pathlib
import random

# Download and extract the dataset
zip_path = tf.keras.utils.get_file(
    fname="fra-eng.zip",
    origin="http://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip",
    extract=True
)

data_path = pathlib.Path(zip_path).parent / "fra.txt"

print(data_path)

3423204/3423204 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
/root/.keras/datasets/fra.txt


In [29]:
import pathlib

dataset_dir = pathlib.Path("/root/.keras/datasets")

for path in dataset_dir.rglob("*"):
    if path.is_file() and ("fra" in path.name.lower() or path.suffix == ".txt"):
        print(path)

/root/.keras/datasets/fra-eng.zip
/root/.keras/datasets/fra-eng_extracted/fra.txt
/root/.keras/datasets/fra-eng_extracted/_about.txt


In [30]:
import pathlib
import random

data_path = pathlib.Path(
    "/root/.keras/datasets/fra-eng_extracted/fra.txt"
)

pairs = []

with open(data_path, encoding="utf-8") as file:
    for line in file:
        parts = line.strip().split("\t")

        if len(parts) >= 2:
            english = parts[0].strip().lower()
            french = parts[1].strip().lower()

            if english and french:
                pairs.append((english, french))

print("Total sentence pairs:", len(pairs))
print("Example English:", pairs[0][0])
print("Example French:", pairs[0][1])

Total sentence pairs: 167130
Example English: go.
Example French: va !


In [31]:
random.seed(42)
random.shuffle(pairs)
pairs = pairs[:20000]

english_sentences = [en for en, fr in pairs]
french_sentences = [fr for en, fr in pairs]

print("Selected pairs:", len(pairs))

Selected pairs: 20000


In [32]:
split = int(0.9 * len(pairs))

train_en = english_sentences[:split]
train_fr = french_sentences[:split]

val_en = english_sentences[split:]
val_fr = french_sentences[split:]

print("Training pairs:", len(train_en))
print("Validation pairs:", len(val_en))

Training pairs: 18000
Validation pairs: 2000


In [33]:
train_fr_marked = [
    "startseq " + sentence + " endseq"
    for sentence in train_fr
]

val_fr_marked = [
    "startseq " + sentence + " endseq"
    for sentence in val_fr
]

print(train_fr_marked[0])

startseq peut-être ne devrions-nous pas faire cela aujourd'hui. endseq


In [34]:
!pip install -q sentencepiece

In [35]:
import sentencepiece as spm
import tensorflow as tf
import numpy as np
import os

In [36]:
os.makedirs("bpe_tokenizers", exist_ok=True)

with open("bpe_tokenizers/english.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(train_en))

with open("bpe_tokenizers/french.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(train_fr))

In [37]:
spm.SentencePieceTrainer.train(
    input="bpe_tokenizers/english.txt",
    model_prefix="bpe_tokenizers/english",
    vocab_size=5000,
    model_type="bpe",
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    character_coverage=1.0
)

spm.SentencePieceTrainer.train(
    input="bpe_tokenizers/french.txt",
    model_prefix="bpe_tokenizers/french",
    vocab_size=5000,
    model_type="bpe",
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    character_coverage=1.0
)

I0000 00:00:1793640988.424201      49 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: bpe_tokenizers/english.txt
  input_format: 
  model_prefix: bpe_tokenizers/english
  model_type: BPE
  vocab_size: 5000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surfac

True

In [38]:
import sentencepiece as spm

source_tokenizer = spm.SentencePieceProcessor(
    model_file="bpe_tokenizers/english.model" 
)

target_tokenizer = spm.SentencePieceProcessor(
    model_file="bpe_tokenizers/french.model"
)

print("English vocabulary:", source_tokenizer.get_piece_size())
print("French vocabulary:", target_tokenizer.get_piece_size())

English vocabulary: 5000
French vocabulary: 5000


In [39]:
import numpy as np

MAX_LEN = 128

def encode_source(sentence):
    ids = source_tokenizer.encode(sentence, out_type=int)
    ids = ids[:MAX_LEN]
    ids += [source_tokenizer.pad_id()] * (MAX_LEN - len(ids))
    return ids


def encode_target(sentence):
    ids = target_tokenizer.encode(sentence, out_type=int)

    ids = (
        [target_tokenizer.bos_id()]
        + ids[:MAX_LEN - 2]
        + [target_tokenizer.eos_id()]
    )

    ids += [target_tokenizer.pad_id()] * (MAX_LEN - len(ids))
    return ids


X_train = np.array([encode_source(s) for s in train_en], dtype=np.int32)
y_train = np.array([encode_target(s) for s in train_fr], dtype=np.int32)

X_val = np.array([encode_source(s) for s in val_en], dtype=np.int32)
y_val = np.array([encode_target(s) for s in val_fr], dtype=np.int32)

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)

Training: (18000, 128) (18000, 128)
Validation: (2000, 128) (2000, 128)


In [59]:
print("Training English:", input_tensor.shape)
print("Training French:", target_tensor.shape)
print("Validation English:", val_input_tensor.shape)
print("Validation French:", val_target_tensor.shape)

Training English: (18000, 128)
Training French: (18000, 128)
Validation English: (2000, 128)
Validation French: (2000, 128)


In [41]:
BATCH_SIZE = 16

train_dataset = (
    tf.data.Dataset
    .from_tensor_slices((input_tensor, target_tensor))
    .shuffle(len(input_tensor), reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    tf.data.Dataset
    .from_tensor_slices((val_input_tensor, val_target_tensor))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [61]:
examples = [
    "je suis un étudiant",
    "comment êtes-vous ?",
    "je adore apprendre le français",
    "où est la gare ?"
]

for sentence in examples:
    pieces = target_tokenizer.encode(
        sentence,
        out_type=int
    )

    print(sentence)
    print(pieces)
    print()

je suis un étudiant
[27, 115, 43, 2517]

comment êtes-vous ?
[297, 204, 4959, 107, 40]

je adore apprendre le français
[27, 2444, 1347, 44, 470]

où est la gare ?
[278, 59, 49, 1098, 40]



In [42]:
transformer = Transformer(
    num_layers=8,
    input_vocab_size=5000,
    embedding_dims=64,
    maximum_position_encoding=128,
    ff_dims=256,
    num_heads=8,
    target_vocab_size=5000
)

In [69]:
print(optimizer)

In [70]:
for input_data, target_data in train_dataset.take(1):

    batch_loss = train_step(
        transformer,
        input_data,
        target_data,
        optimizer
    )

    print("Batch loss:", float(batch_loss.numpy()))

Batch loss: 8.548380851745605


In [71]:
train_loss = tf.keras.metrics.Mean(name="train_loss")

In [46]:
BATCH_SIZE = 16

train_dataset = (
    tf.data.Dataset
    .from_tensor_slices((X_train, y_train))
    .shuffle(20000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [74]:
import tensorflow as tf

print("Available GPUs:")

for gpu in tf.config.list_physical_devices('GPU'):
    print(gpu)

print("\nGPU count:",
      len(tf.config.list_physical_devices('GPU')))

Available GPUs:
PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')

GPU count: 2


In [47]:
strategy = tf.distribute.MirroredStrategy()

print("Number of devices:", strategy.num_replicas_in_sync)

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Number of devices: 2


In [76]:
with strategy.scope():

    transformer = Transformer(
        num_layers=8,
        input_vocab_size=5000,
        embedding_dims=64,
        maximum_position_encoding=128,
        ff_dims=256,
        num_heads=8,
        target_vocab_size=5000
    )

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=0.0001,
        beta_1=0.9,
        beta_2=0.98,
        epsilon=1e-9
    )

    train_loss = tf.keras.metrics.Mean(name="train_loss")

In [77]:
import inspect

print(inspect.signature(Transformer.__init__))

(self, num_layers, input_vocab_size, embedding_dims, maximum_position_encoding, ff_dims, num_heads, target_vocab_size)


In [78]:
dummy_input = tf.zeros((1, MAX_LEN), dtype=tf.int32)
dummy_target = tf.zeros((1, MAX_LEN), dtype=tf.int32)

with strategy.scope():
    _ = transformer(
        dummy_input,
        dummy_target,
        create_padding_mask(dummy_input),
        create_lookahead_mask(MAX_LEN),
        create_padding_mask(dummy_input),
        training=False
    )

    optimizer.build(transformer.trainable_variables)

print("Model initialized successfully!")
print("Trainable parameters:", transformer.count_params())
print("GPUs:", strategy.num_replicas_in_sync)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder__layer_22' (of type Encoder_Layer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder__layer_23' (of type Encoder_Layer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder__layer_24' (of type Encoder_Layer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn

Model initialized successfully!
Trainable parameters: 1898888
GPUs: 2


In [48]:
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True,
    reduction=tf.keras.losses.Reduction.NONE
)


@tf.function
def distributed_train_step(input_data, target_data):

    def replica_step(input_data, target_data):

        dec_input = target_data[:, :-1]
        dec_real = target_data[:, 1:]

        enc_padding_mask = create_padding_mask(input_data)
        target_padding_mask = create_padding_mask(dec_input)

        target_length = tf.shape(dec_input)[1]

        look_ahead_mask = create_lookahead_mask(target_length)

        combined_mask = tf.logical_and(
            target_padding_mask,
            look_ahead_mask
        )

        with tf.GradientTape() as tape:

            predictions = transformer(
                input_data,
                dec_input,
                enc_padding_mask,
                combined_mask,
                enc_padding_mask,
                training=True
            )

            token_losses = loss_object(dec_real, predictions)

            mask = tf.cast(
                tf.not_equal(dec_real, 0),
                token_losses.dtype
            )

            local_loss_sum = tf.reduce_sum(token_losses * mask)
            local_token_count = tf.reduce_sum(mask)

            replica_context = tf.distribute.get_replica_context()

            global_token_count = replica_context.all_reduce(
                tf.distribute.ReduceOp.SUM,
                local_token_count
            )

            loss = local_loss_sum / global_token_count

        gradients = tape.gradient(
            loss,
            transformer.trainable_variables
        )

        optimizer.apply_gradients(
            zip(gradients, transformer.trainable_variables)
        )

        return loss

    per_replica_loss = strategy.run(
        replica_step,
        args=(input_data, target_data)
    )

    return strategy.reduce(
        tf.distribute.ReduceOp.SUM,
        per_replica_loss,
        axis=None
    )

In [49]:
distributed_train_dataset = strategy.experimental_distribute_dataset(
    train_dataset
)

print("Distributed dataset created successfully!")

Distributed dataset created successfully!


In [ ]:
EPOCHS = 15

for epoch in range(EPOCHS):

    train_loss.reset_state()

    for input_data, target_data in distributed_train_dataset:

        batch_loss = distributed_train_step(
            input_data,
            target_data
        )

        train_loss.update_state(batch_loss)

    epoch_loss = train_loss.result().numpy()

    print(
        f"Epoch {epoch + 1}/{EPOCHS} - "
        f"Loss: {epoch_loss:.4f}"
    )

    transformer.save_weights(
        f"transformer_100k_epoch_{epoch + 1}.weights.h5"
    )

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1


In [82]:
distributed_val_dataset = strategy.experimental_distribute_dataset(
    val_dataset
)

In [50]:
@tf.function
def distributed_val_step(input_data, target_data):

    def replica_step(input_data, target_data):

        dec_input = target_data[:, :-1]
        dec_real = target_data[:, 1:]

        enc_padding_mask = create_padding_mask(input_data)
        target_padding_mask = create_padding_mask(dec_input)

        look_ahead_mask = create_lookahead_mask(
            tf.shape(dec_input)[1]
        )

        combined_mask = tf.logical_and(
            target_padding_mask,
            look_ahead_mask
        )

        predictions = transformer(
            input_data,
            dec_input,
            enc_padding_mask,
            combined_mask,
            enc_padding_mask,
            training=False
        )

        token_losses = loss_object(dec_real, predictions)

        mask = tf.cast(
            tf.not_equal(dec_real, 0),
            token_losses.dtype
        )

        local_loss_sum = tf.reduce_sum(token_losses * mask)
        local_token_count = tf.reduce_sum(mask)

        context = tf.distribute.get_replica_context()

        global_token_count = context.all_reduce(
            tf.distribute.ReduceOp.SUM,
            local_token_count
        )

        return local_loss_sum / global_token_count

    per_replica_loss = strategy.run(
        replica_step,
        args=(input_data, target_data)
    )

    return strategy.reduce(
        tf.distribute.ReduceOp.SUM,
        per_replica_loss,
        axis=None
    )

In [51]:
def validate():

    val_loss = tf.keras.metrics.Mean()

    for input_data, target_data in distributed_val_dataset:

        batch_loss = distributed_val_step(
            input_data,
            target_data
        )

        val_loss.update_state(batch_loss)

    return val_loss.result().numpy()

In [87]:
EPOCHS = 15

for epoch in range(EPOCHS):

    train_loss.reset_state()

    for input_data, target_data in distributed_train_dataset:

        batch_loss = distributed_train_step(
            input_data,
            target_data
        )

        train_loss.update_state(batch_loss)

    epoch_train_loss = train_loss.result().numpy()

    epoch_val_loss = validate()

    print(
        f"Epoch {epoch + 1} | "
        f"Training Loss: {epoch_train_loss:.4f} | "
        f"Validation Loss: {epoch_val_loss:.4f}"
    )

    transformer.save_weights(
        f"transformer_5k_epoch_{epoch + 1}.weights.h5"
    )

INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
Epoch 1 | Training Loss: 4.7400 | Validation Loss: 4.5374
Epoch 2 | Training Loss: 4.4069 | Validation Loss: 4.2909
Epoch 3 | Training Loss: 4.1944 | Validation Loss: 4.1021
Epoch 4 | Training Loss: 4.0306 | Validation Loss: 3.9576
Epoch 5 | Training Loss: 3.9035 | Validation Loss: 3.8407
Epoch 6 | Training Loss: 3.7925 | Validation Loss: 3.7436
Epoch 7 | Training Loss: 3.6979 | Validation Loss: 3.6603
Epoch 8 | Training Loss: 3.6112 | Validation Loss: 3.5734
Epoch 9 | Training Loss: 3.5322 | Validation Loss: 3.5015
Epoch 10 | Training Loss: 3.4561 | Validation Loss: 3.4324
Epoch 11 | Training Loss: 3.3870 | Validation Loss: 3.3617
Epoch 12 | Training Loss: 3.3247 | Validation Loss: 3.3004
Epoch 13 | Training Loss: 3.2613 | Validation Loss: 3.2444
Epoch 14 | Training Loss: 3.2024 | Validation Loss: 3.1880
Epoch 15 | Training Los

In [88]:
transformer.save_weights(
    "transformer_20k_epoch_15.weights.h5"
)

In [89]:
EPOCHS = 15

for epoch in range(EPOCHS):

    train_loss.reset_state()

    for input_data, target_data in distributed_train_dataset:
        batch_loss = distributed_train_step(
            input_data,
            target_data
        )
        train_loss.update_state(batch_loss)

    epoch_train_loss = train_loss.result().numpy()
    epoch_val_loss = validate()

    print(
        f"Epoch {epoch + 16} | "
        f"Training Loss: {epoch_train_loss:.4f} | "
        f"Validation Loss: {epoch_val_loss:.4f}"
    )

    transformer.save_weights(
        f"transformer_20k_epoch_{epoch + 16}.weights.h5"
    )

Epoch 16 | Training Loss: 3.0966 | Validation Loss: 3.0991
Epoch 17 | Training Loss: 3.0495 | Validation Loss: 3.0475
Epoch 18 | Training Loss: 3.0020 | Validation Loss: 3.0172
Epoch 19 | Training Loss: 2.9552 | Validation Loss: 2.9670
Epoch 20 | Training Loss: 2.9128 | Validation Loss: 2.9254
Epoch 21 | Training Loss: 2.8718 | Validation Loss: 2.8932
Epoch 22 | Training Loss: 2.8338 | Validation Loss: 2.8713
Epoch 23 | Training Loss: 2.7924 | Validation Loss: 2.8307
Epoch 24 | Training Loss: 2.7580 | Validation Loss: 2.8008
Epoch 25 | Training Loss: 2.7203 | Validation Loss: 2.7710
Epoch 26 | Training Loss: 2.6889 | Validation Loss: 2.7484
Epoch 27 | Training Loss: 2.6542 | Validation Loss: 2.7134
Epoch 28 | Training Loss: 2.6216 | Validation Loss: 2.6938
Epoch 29 | Training Loss: 2.5888 | Validation Loss: 2.6696
Epoch 30 | Training Loss: 2.5609 | Validation Loss: 2.6442


In [90]:
EPOCHS = 15

for epoch in range(EPOCHS):

    train_loss.reset_state()

    for input_data, target_data in distributed_train_dataset:

        batch_loss = distributed_train_step(
            input_data,
            target_data
        )

        train_loss.update_state(batch_loss)

    epoch_train_loss = train_loss.result().numpy()
    epoch_val_loss = validate()

    current_epoch = epoch + 31

    print(
        f"Epoch {current_epoch} | "
        f"Training Loss: {epoch_train_loss:.4f} | "
        f"Validation Loss: {epoch_val_loss:.4f}"
    )

    transformer.save_weights(
        f"transformer_20k_epoch_{current_epoch}.weights.h5"
    )

Epoch 31 | Training Loss: 2.5315 | Validation Loss: 2.6247
Epoch 32 | Training Loss: 2.5040 | Validation Loss: 2.6010
Epoch 33 | Training Loss: 2.4757 | Validation Loss: 2.5833
Epoch 34 | Training Loss: 2.4484 | Validation Loss: 2.5593
Epoch 35 | Training Loss: 2.4242 | Validation Loss: 2.5422
Epoch 36 | Training Loss: 2.3999 | Validation Loss: 2.5276
Epoch 37 | Training Loss: 2.3744 | Validation Loss: 2.5190
Epoch 38 | Training Loss: 2.3537 | Validation Loss: 2.5115
Epoch 39 | Training Loss: 2.3288 | Validation Loss: 2.4838
Epoch 40 | Training Loss: 2.3069 | Validation Loss: 2.4745
Epoch 41 | Training Loss: 2.2892 | Validation Loss: 2.4562
Epoch 42 | Training Loss: 2.2672 | Validation Loss: 2.4497
Epoch 43 | Training Loss: 2.2442 | Validation Loss: 2.4330
Epoch 44 | Training Loss: 2.2279 | Validation Loss: 2.4263
Epoch 45 | Training Loss: 2.2130 | Validation Loss: 2.4111


In [91]:
transformer.load_weights(
    "transformer_20k_epoch_45.weights.h5"
)

In [57]:
def translate(sentence, max_length=128):

    # Encode English sentence
    input_ids = source_tokenizer.encode(
        sentence,
        out_type=int
    )

    input_ids = input_ids[:max_length]

    inputs = tf.constant([input_ids], dtype=tf.int32)

    # Encoder padding mask
    enc_padding_mask = create_padding_mask(inputs)

    # Encode source sentence
    enc_output = transformer.encoder(
        inputs,
        training=False,
        mask=enc_padding_mask
    )

    # Start French translation with BOS
    output = [target_tokenizer.bos_id()]

    for _ in range(max_length - 1):

        dec_input = tf.constant([output], dtype=tf.int32)

        # Decoder self-attention mask
        look_ahead_mask = create_lookahead_mask(
            tf.shape(dec_input)[1]
        )

        # Decoder cross-attention mask
        dec_padding_mask = create_padding_mask(inputs)

        # Decoder
        dec_output = transformer.decoder(
            dec_input,
            enc_output,
            training=False,
            padding_mask=dec_padding_mask,
            look_ahead_mask=look_ahead_mask
        )

        # Final Dense layer
        predictions = transformer.final_layer(dec_output)

        # Select next token
        predicted_id = int(
            tf.argmax(predictions[0, -1, :]).numpy()
        )

        # Stop at EOS
        if predicted_id == target_tokenizer.eos_id():
            break

        output.append(predicted_id)

    # Convert generated token IDs to French text
    translation = target_tokenizer.decode(output[1:])

    return translation

In [99]:
# Test one sentence from training data
print("TRAINING EXAMPLE")
print("English:", train_en[0])
print("Actual French:", train_fr[0])
print("Predicted:", translate(train_en[0]))

print("\nVALIDATION EXAMPLE")
print("English:", val_en[0])
print("Actual French:", val_fr[0])
print("Predicted:", translate(val_en[0]))

TRAINING EXAMPLE
English: maybe we shouldn't do that today.
Actual French: peut-être ne devrions-nous pas faire cela aujourd'hui.
Predicted: peut-être ne devrions-nous pas faire ça aujourd'hui.

VALIDATION EXAMPLE
English: the joke's on you.
Actual French: on se fout de ta gueule.
Predicted: la police est sur toi.


In [100]:
# Inspect tokenization and decoding

for i in range(5):

    en = train_en[i]
    fr = train_fr[i]

    en_ids = source_tokenizer.encode(en)
    fr_ids = target_tokenizer.encode(fr)

    print("English:", en)
    print("English tokens:", len(en_ids))
    print("English decoded:", source_tokenizer.decode(en_ids))

    print("French:", fr)
    print("French tokens:", len(fr_ids))
    print("French decoded:", target_tokenizer.decode(fr_ids))

    print("-" * 60)

English: maybe we shouldn't do that today.
English tokens: 9
English decoded: maybe we shouldn't do that today.
French: peut-être ne devrions-nous pas faire cela aujourd'hui.
French tokens: 14
French decoded: peut-être ne devrions-nous pas faire cela aujourd'hui.
------------------------------------------------------------
English: let's hurry.
English tokens: 5
English decoded: let's hurry.
French: dépêchons-nous.
French tokens: 5
French decoded: dépêchons-nous.
------------------------------------------------------------
English: did you already do your homework?
English tokens: 7
English decoded: did you already do your homework?
French: as-tu déjà fini tes devoirs ?
French tokens: 8
French decoded: as-tu déjà fini tes devoirs ?
------------------------------------------------------------
English: my mother has been dead these ten years.
English tokens: 9
English decoded: my mother has been dead these ten years.
French: ma mère est décédée depuis dix ans.
French tokens: 8
French dec

In [101]:
import random

random.seed(42)

print("========== TRAINING ==========")

for i in random.sample(range(len(train_en)), 10):
    prediction = translate(train_en[i])

    print("\nEnglish:", train_en[i])
    print("Actual: ", train_fr[i])
    print("Predicted:", prediction)


print("\n========== VALIDATION ==========")

for i in random.sample(range(len(val_en)), 10):
    prediction = translate(val_en[i])

    print("\nEnglish:", val_en[i])
    print("Actual: ", val_fr[i])
    print("Predicted:", prediction)

========== TRAINING ==========

English: you can't leave me now.
Actual:  vous ne pouvez pas me laisser maintenant.
Predicted: tu ne peux pas me laisser maintenant.

English: i'm a management consultant.
Actual:  je suis conseiller en gestion.
Predicted: je suis un coup de me faire.

English: i usually shower at night.
Actual:  d'ordinaire, je me douche le soir.
Predicted: je me suis senti en japonais.

English: i don't like that one.
Actual:  je n'aime pas celui-là.
Predicted: je n'aime pas ça.

English: do you like singing?
Actual:  aimez-vous chanter ?
Predicted: aimez-vous chanter ?

English: stop it right now!
Actual:  arrête immédiatement !
Predicted: arrête de le faire maintenant !

English: i enjoyed every minute of it.
Actual:  j'ai joui de chaque minute.
Predicted: je me suis senti tous les minutes.

English: let's dispense with formalities.
Actual:  passons-nous des formalités.
Predicted: faisons desvouss de se produire.

English: are you surprised?
Actual:  es-tu surpris ?


In [110]:
def evaluate_token_accuracy(X, Y, batch_size=64):

    total_correct = 0
    total_tokens = 0

    for start in range(0, len(X), batch_size):

        end = start + batch_size

        inputs = tf.constant(X[start:end], dtype=tf.int32)
        targets = tf.constant(Y[start:end], dtype=tf.int32)

        dec_input = targets[:, :-1]
        actual = targets[:, 1:]

        enc_mask = create_padding_mask(inputs)
        dec_mask = create_padding_mask(inputs)

        look_ahead = create_lookahead_mask(
            tf.shape(dec_input)[1]
        )

        logits = transformer(
            inputs,
            dec_input,
            enc_mask,
            look_ahead,
            dec_mask,
            training=False
        )

        predicted = tf.argmax(
            logits,
            axis=-1,
            output_type=tf.int32
        )

        valid_tokens = tf.not_equal(actual, 0)

        correct = tf.logical_and(
            tf.equal(predicted, actual),
            valid_tokens
        )

        total_correct += tf.reduce_sum(
            tf.cast(correct, tf.int64)
        ).numpy()

        total_tokens += tf.reduce_sum(
            tf.cast(valid_tokens, tf.int64)
        ).numpy()

    return total_correct / total_tokens

In [111]:
print(
    "Training token accuracy:",
    evaluate_token_accuracy(X_train, y_train)
)

print(
    "Validation token accuracy:",
    evaluate_token_accuracy(X_val, y_val)
)

Training token accuracy: 0.6534522424712421
Validation token accuracy: 0.5854764820610514


In [107]:
# Select a training example
inputs = tf.constant(X_train[:1], dtype=tf.int32)
targets = tf.constant(y_train[:1], dtype=tf.int32)

# Teacher forcing
dec_input = targets[:, :-1]
actual_ids = targets[:, 1:]

# Create masks
enc_padding_mask = create_padding_mask(inputs)
dec_padding_mask = create_padding_mask(inputs)

look_ahead_mask = create_lookahead_mask(
    tf.shape(dec_input)[1]
)

# Forward pass
predictions = transformer(
    inputs,
    dec_input,
    enc_padding_mask,
    look_ahead_mask,
    dec_padding_mask,
    training=False
)

# Predicted token IDs (force int32)
predicted_ids = tf.argmax(
    predictions,
    axis=-1,
    output_type=tf.int32
)

# Ignore padding tokens
valid_tokens = tf.not_equal(actual_ids, 0)

correct_tokens = tf.logical_and(
    tf.equal(predicted_ids, actual_ids),
    valid_tokens
)

accuracy = (
    tf.reduce_sum(tf.cast(correct_tokens, tf.float32))
    / tf.reduce_sum(tf.cast(valid_tokens, tf.float32))
)

print("Teacher-forced token accuracy:", accuracy.numpy())

Teacher-forced token accuracy: 0.93333334


In [115]:
def beam_search_translate(sentence, beam_width=5, max_length=128):

    # Encode English sentence
    input_ids = source_tokenizer.encode(sentence, out_type=int)
    input_ids = input_ids[:max_length]

    inputs = tf.constant([input_ids], dtype=tf.int32)

    # Encode source sentence once
    enc_padding_mask = create_padding_mask(inputs)

    enc_output = transformer.encoder(
        inputs,
        training=False,
        mask=enc_padding_mask
    )

    bos_id = target_tokenizer.bos_id()
    eos_id = target_tokenizer.eos_id()

    # Each beam: (token sequence, cumulative log probability)
    beams = [([bos_id], 0.0)]

    for step in range(max_length - 1):

        candidates = []

        for sequence, score in beams:

            # Preserve completed sequences
            if sequence[-1] == eos_id:
                candidates.append((sequence, score))
                continue

            dec_input = tf.constant(
                [sequence],
                dtype=tf.int32
            )

            look_ahead_mask = create_lookahead_mask(
                tf.shape(dec_input)[1]
            )

            dec_padding_mask = create_padding_mask(inputs)

            # Decoder forward pass
            dec_output = transformer.decoder(
                dec_input,
                enc_output,
                training=False,
                padding_mask=dec_padding_mask,
                look_ahead_mask=look_ahead_mask
            )

            logits = transformer.final_layer(dec_output)

            # Last position prediction
            log_probs = tf.nn.log_softmax(
                logits[0, -1, :]
            )

            # Select top beam_width next tokens
            top_log_probs, top_ids = tf.math.top_k(
                log_probs,
                k=beam_width
            )

            for log_prob, token_id in zip(
                top_log_probs.numpy(),
                top_ids.numpy()
            ):

                new_sequence = sequence + [int(token_id)]

                new_score = score + float(log_prob)

                candidates.append(
                    (new_sequence, new_score)
                )

        # Keep the best candidates
        candidates.sort(
            key=lambda item: item[1],
            reverse=True
        )

        beams = candidates[:beam_width]

        # Stop when all beams have reached EOS
        if all(seq[-1] == eos_id for seq, _ in beams):
            break

    # Select the best completed beam if available
    completed = [
        beam for beam in beams
        if beam[0][-1] == eos_id
    ]

    best_sequence, best_score = max(
        completed if completed else beams,
        key=lambda item: item[1]
    )

    # Remove BOS and EOS
    output_ids = best_sequence[1:]

    if output_ids and output_ids[-1] == eos_id:
        output_ids = output_ids[:-1]

    return target_tokenizer.decode(output_ids)

In [117]:
sample = target_tensor[0]

print("Full target:")
print(sample[:20])

print("\nDecoder input:")
print(sample[:-1][:20])

print("\nExpected output:")
print(sample[1:][:20])

print("\nDecoded target:")
print(target_tokenizer.decode(
    sample[1:][sample[1:] != 0]
))

Full target:
[   2  331 4959  354   47 1190 4959  496   37  124  232  490 4953  471
 4949    3    0    0    0    0]

Decoder input:
[   2  331 4959  354   47 1190 4959  496   37  124  232  490 4953  471
 4949    3    0    0    0    0]

Expected output:
[ 331 4959  354   47 1190 4959  496   37  124  232  490 4953  471 4949
    3    0    0    0    0    0]

Decoded target:
peut-être ne devrions-nous pas faire cela aujourd'hui.


In [119]:
transformer.save_weights(
    "transformer_20k_original_epoch45.weights.h5"
)

In [120]:
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", optimizer.learning_rate.numpy())
print("Beta 1:", optimizer.beta_1)
print("Beta 2:", optimizer.beta_2)
print("Epsilon:", optimizer.epsilon)

Optimizer: Adam
Learning rate: 1e-04
Beta 1: 0.9
Beta 2: 0.98
Epsilon: 1e-09


In [123]:
import os

for root, dirs, files in os.walk("/kaggle"):
    for file in files:
        if file.endswith(".weights.h5"):
            print(os.path.join(root, file))

/kaggle/working/transformer_20k_epoch_36.weights.h5
/kaggle/working/transformer_20k_epoch_38.weights.h5
/kaggle/working/transformer_20k_epoch_20.weights.h5
/kaggle/working/transformer_20k_epoch_30.weights.h5
/kaggle/working/transformer_20k_epoch_25.weights.h5
/kaggle/working/transformer_20k_epoch_26.weights.h5
/kaggle/working/transformer_20k_epoch_22.weights.h5
/kaggle/working/transformer_20k_epoch_23.weights.h5
/kaggle/working/transformer_5k_epoch_14.weights.h5
/kaggle/working/transformer_20k_epoch_43.weights.h5
/kaggle/working/transformer_20k_epoch_35.weights.h5
/kaggle/working/transformer_20k_epoch_39.weights.h5
/kaggle/working/transformer_5k_epoch_13.weights.h5
/kaggle/working/transformer_20k_epoch_34.weights.h5
/kaggle/working/transformer_20k_epoch_21.weights.h5
/kaggle/working/transformer_20k_original_epoch45.weights.h5
/kaggle/working/transformer_20k_epoch_27.weights.h5
/kaggle/working/transformer_5k_epoch_2.weights.h5
/kaggle/working/transformer_20k_epoch_29.weights.h5
/kaggle/

In [131]:
print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__)

TensorFlow: 2.20.0
Keras: 3.13.2


In [133]:
import os

checkpoint = "/kaggle/working/transformer_20k_original_epoch45.weights.h5"

print("Checkpoint exists:", os.path.exists(checkpoint))
print("Checkpoint size:", os.path.getsize(checkpoint) / (1024**2), "MB")

Checkpoint exists: True
Checkpoint size: 7.93719482421875 MB


In [134]:
EPOCHS = 15

for epoch in range(EPOCHS):

    train_loss.reset_state()

    for input_data, target_data in distributed_train_dataset:

        batch_loss = distributed_train_step(
            input_data,
            target_data
        )

        train_loss.update_state(batch_loss)

    epoch_train_loss = train_loss.result().numpy()
    epoch_val_loss = validate()

    current_epoch = epoch + 46

    print(
        f"Epoch {current_epoch} | "
        f"Training Loss: {epoch_train_loss:.4f} | "
        f"Validation Loss: {epoch_val_loss:.4f}"
    )

    transformer.save_weights(
        f"transformer_20k_adam_epoch_{current_epoch}.weights.h5"
    )

Epoch 46 | Training Loss: 2.1947 | Validation Loss: 2.3989
Epoch 47 | Training Loss: 2.1786 | Validation Loss: 2.3916
Epoch 48 | Training Loss: 2.1612 | Validation Loss: 2.3862
Epoch 49 | Training Loss: 2.1467 | Validation Loss: 2.3723
Epoch 50 | Training Loss: 2.1275 | Validation Loss: 2.3680
Epoch 51 | Training Loss: 2.1157 | Validation Loss: 2.3614
Epoch 52 | Training Loss: 2.1051 | Validation Loss: 2.3544
Epoch 53 | Training Loss: 2.0885 | Validation Loss: 2.3393
Epoch 54 | Training Loss: 2.0747 | Validation Loss: 2.3349
Epoch 55 | Training Loss: 2.0621 | Validation Loss: 2.3281
Epoch 56 | Training Loss: 2.0438 | Validation Loss: 2.3241
Epoch 57 | Training Loss: 2.0378 | Validation Loss: 2.3206
Epoch 58 | Training Loss: 2.0237 | Validation Loss: 2.3071
Epoch 59 | Training Loss: 2.0094 | Validation Loss: 2.3017
Epoch 60 | Training Loss: 1.9992 | Validation Loss: 2.2919


In [137]:
test_sentences = [
    "Hello, how are you?",
    "I am a student.",
    "I love learning French.",
    "Where are you going?",
    "This is a beautiful day.",
    "I didn't see what was in the box.",
    "I want to learn artificial intelligence.",
    "She is reading a book.",
    "We should work together.",
    "I hope everything goes well."
]

for sentence in test_sentences:

    translation = translate(
        sentence,
        max_length=50
    )

    print("English:", sentence)
    print("French: ", translation)
    print("-" * 60)

English: Hello, how are you?
French:  l'alono est-il blessé, si ?
------------------------------------------------------------
English: I am a student.
French:  le bis est un journal.
------------------------------------------------------------
English: I love learning French.
French:  l'amour a fait l'air l'air l'air réle.
------------------------------------------------------------
English: Where are you going?
French:  l'as-tuilaslé !
------------------------------------------------------------
English: This is a beautiful day.
French:  le tanbon est un jour de jour.
------------------------------------------------------------
English: I didn't see what was in the box.
French:  l'alonage ne s'est pas vu dans la boîte.
------------------------------------------------------------
English: I want to learn artificial intelligence.
French:  le catbon veut apprendre à apprendre des caux.
------------------------------------------------------------
English: She is reading a book.
French:  

In [140]:
# Select one validation example
index = 567

source_ids = X_val[index:index+1]
target_ids = y_val[index:index+1]

# Decoder input and expected output
dec_input = target_ids[:, :-1]
dec_real = target_ids[:, 1:]

# Masks
enc_mask = create_padding_mask(source_ids)
target_mask = create_padding_mask(dec_input)

look_mask = create_lookahead_mask(
    tf.shape(dec_input)[1]
)

combined_mask = tf.logical_and(
    target_mask,
    look_mask
)

# Teacher-forced prediction
predictions = transformer(
    source_ids,
    dec_input,
    enc_mask,
    combined_mask,
    enc_mask,
    training=False
)

predicted_ids = tf.argmax(
    predictions,
    axis=-1
).numpy()[0]

actual_ids = dec_real[0]
# Ignore padding
valid_positions = actual_ids != 0

accuracy = (
    (predicted_ids[valid_positions] == actual_ids[valid_positions])
    .mean()
)

print("Teacher-forced token accuracy:", accuracy)

print("\nExpected French:")
print(target_tokenizer.decode(
    [int(x) for x in actual_ids if x not in [0, 3]]
))

print("\nPredicted French:")
print(target_tokenizer.decode(
    [int(x) for x in predicted_ids[valid_positions] if x != 3]
))

Teacher-forced token accuracy: 0.7272727272727273

Expected French:
je veux juste voir ce qui se passera.

Predicted French:
je veux juste voir ce qui va passe..


In [141]:
correct_tokens = 0
total_tokens = 0

BATCH_SIZE = 64

for start in range(0, len(X_val), BATCH_SIZE):

    source_ids = X_val[start:start+BATCH_SIZE]
    target_ids = y_val[start:start+BATCH_SIZE]

    dec_input = target_ids[:, :-1]
    dec_real = target_ids[:, 1:]

    enc_mask = create_padding_mask(source_ids)
    target_mask = create_padding_mask(dec_input)

    look_mask = create_lookahead_mask(
        tf.shape(dec_input)[1]
    )

    combined_mask = tf.logical_and(
        target_mask,
        look_mask
    )

    predictions = transformer(
        source_ids,
        dec_input,
        enc_mask,
        combined_mask,
        enc_mask,
        training=False
    )

    predicted_ids = tf.argmax(
        predictions,
        axis=-1
    ).numpy()

    actual_ids = dec_real

    valid_mask = actual_ids != 0

    correct_tokens += (
        (predicted_ids == actual_ids) & valid_mask
    ).sum()

    total_tokens += valid_mask.sum()

accuracy = correct_tokens / total_tokens

print("Full validation token accuracy:", accuracy)

Full validation token accuracy: 0.6096521068626117


In [ ]:
import os
import json
import shutil

SAVE_DIR = "/kaggle/working/english_french_transformer"

os.makedirs(SAVE_DIR, exist_ok=True)

# Save current model weights
transformer.save_weights(
    os.path.join(SAVE_DIR, "epoch_57.weights.h5")
)

# Save training configuration
config = {
    "model_type": "Transformer",
    "source_language": "English",
    "target_language": "French",
    "num_layers": 8,
    "embedding_dims": 64,
    "num_heads": 8,
    "ff_dims": 256,
    "maximum_position_encoding": 128,
    "source_vocab_size": 5000,
    "target_vocab_size": 5000,
    "max_sequence_length": 128,
    "optimizer": "Adam",
    "learning_rate": 1e-4,
    "beta_1": 0.9,
    "beta_2": 0.98,
    "epsilon": 1e-9,
    "completed_epochs": 57
}

with open(os.path.join(SAVE_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=4)

print("Model weights and configuration saved!")

In [147]:
with strategy.scope():
    transformer_fast = Transformer(
        num_layers=8,
        input_vocab_size=5000,
        embedding_dims=64,
        maximum_position_encoding=128,
        ff_dims=256,
        num_heads=8,
        target_vocab_size=5000
    )

    # Build the model
    dummy_input = tf.zeros((1, MAX_LEN), dtype=tf.int32)
    dummy_target = tf.zeros((1, MAX_LEN - 1), dtype=tf.int32)

    transformer_fast(
        dummy_input,
        dummy_target,
        create_padding_mask(dummy_input),
        tf.logical_and(
            create_padding_mask(dummy_target),
            create_lookahead_mask(MAX_LEN - 1)
        ),
        create_padding_mask(dummy_input),
        training=False
    )

    # Load epoch-60 weights
    transformer_fast.load_weights(
        "/kaggle/working/english_french_transformer_backup/model_epoch_60.weights.h5"
    )

print("Separate model created and epoch-60 weights loaded.")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder__layer_78' (of type Encoder_Layer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder__layer_79' (of type Encoder_Layer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder__layer_80' (of type Encoder_Layer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn

Separate model created and epoch-60 weights loaded.


In [148]:
with strategy.scope():
    optimizer_fast = tf.keras.optimizers.Adam(
        learning_rate=1.5e-4,
        beta_1=0.9,
        beta_2=0.98,
        epsilon=1e-9
    )

    optimizer_fast.build(
        transformer_fast.trainable_variables
    )

print("Adam optimizer initialized with LR = 1.5e-4")

Adam optimizer initialized with LR = 1.5e-4


In [149]:
@tf.function
def distributed_train_step_fast(input_data, target_data):

    def replica_step(input_data, target_data):
        dec_input = target_data[:, :-1]
        dec_real = target_data[:, 1:]

        enc_padding_mask = create_padding_mask(input_data)
        target_padding_mask = create_padding_mask(dec_input)

        target_length = tf.shape(dec_input)[1]
        look_ahead_mask = create_lookahead_mask(target_length)

        combined_mask = tf.logical_and(
            target_padding_mask,
            look_ahead_mask
        )

        with tf.GradientTape() as tape:
            predictions = transformer_fast(
                input_data,
                dec_input,
                enc_padding_mask,
                combined_mask,
                enc_padding_mask,
                training=True
            )

            token_losses = loss_object(dec_real, predictions)

            mask = tf.cast(
                tf.not_equal(dec_real, 0),
                token_losses.dtype
            )

            local_loss_sum = tf.reduce_sum(token_losses * mask)
            local_token_count = tf.reduce_sum(mask)

            replica_context = tf.distribute.get_replica_context()

            global_token_count = replica_context.all_reduce(
                tf.distribute.ReduceOp.SUM,
                local_token_count
            )

            loss = local_loss_sum / global_token_count

        gradients = tape.gradient(
            loss,
            transformer_fast.trainable_variables
        )

        optimizer_fast.apply_gradients(
            zip(gradients, transformer_fast.trainable_variables)
        )

        return loss

    per_replica_loss = strategy.run(
        replica_step,
        args=(input_data, target_data)
    )

    return strategy.reduce(
        tf.distribute.ReduceOp.SUM,
        per_replica_loss,
        axis=None
    )

print("Separate distributed training step is ready.")

Separate distributed training step is ready.


In [151]:
def validate_fast():
    val_loss = tf.keras.metrics.Mean()

    for input_data, target_data in distributed_val_dataset:
        batch_loss = distributed_val_step_fast(
            input_data,
            target_data
        )
        val_loss.update_state(batch_loss)

    return val_loss.result().numpy()

In [152]:
@tf.function
def distributed_val_step_fast(input_data, target_data):

    def replica_step(input_data, target_data):
        dec_input = target_data[:, :-1]
        dec_real = target_data[:, 1:]

        enc_padding_mask = create_padding_mask(input_data)
        target_padding_mask = create_padding_mask(dec_input)

        look_ahead_mask = create_lookahead_mask(
            tf.shape(dec_input)[1]
        )

        combined_mask = tf.logical_and(
            target_padding_mask,
            look_ahead_mask
        )

        predictions = transformer_fast(
            input_data,
            dec_input,
            enc_padding_mask,
            combined_mask,
            enc_padding_mask,
            training=False
        )

        token_losses = loss_object(dec_real, predictions)
        mask = tf.cast(tf.not_equal(dec_real, 0), token_losses.dtype)

        local_loss_sum = tf.reduce_sum(token_losses * mask)
        local_token_count = tf.reduce_sum(mask)

        replica_context = tf.distribute.get_replica_context()

        global_token_count = replica_context.all_reduce(
            tf.distribute.ReduceOp.SUM,
            local_token_count
        )

        return local_loss_sum / global_token_count

    per_replica_loss = strategy.run(
        replica_step,
        args=(input_data, target_data)
    )

    return strategy.reduce(
        tf.distribute.ReduceOp.SUM,
        per_replica_loss,
        axis=None
    )

In [153]:
EPOCHS_FAST = 5

for epoch in range(EPOCHS_FAST):

    train_loss.reset_state()

    for input_data, target_data in distributed_train_dataset:
        batch_loss = distributed_train_step_fast(
            input_data,
            target_data
        )
        train_loss.update_state(batch_loss)

    epoch_train_loss = train_loss.result().numpy()
    epoch_val_loss = validate_fast()

    print(
        f"Fast LR Epoch {epoch + 1} | "
        f"Training Loss: {epoch_train_loss:.4f} | "
        f"Validation Loss: {epoch_val_loss:.4f}"
    )

    transformer_fast.save_weights(
        f"/kaggle/working/transformer_fast_lr_epoch_{epoch + 1}.weights.h5"
    )

INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1


INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1


Fast LR Epoch 1 | Training Loss: 2.0020 | Validation Loss: 2.3034
Fast LR Epoch 2 | Training Loss: 1.9854 | Validation Loss: 2.2852
Fast LR Epoch 3 | Training Loss: 1.9686 | Validation Loss: 2.2730
Fast LR Epoch 4 | Training Loss: 1.9519 | Validation Loss: 2.2699
Fast LR Epoch 5 | Training Loss: 1.9406 | Validation Loss: 2.2855


In [30]:
import requests
import base64
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")

repo = "NaveedAhmad-6522/Machine-Learning"

path = (
    "Natural-Language-Processing/"
    "English-French-Transformer/"
    "English-French-Transformer.ipynb"
)

local_path = f"/kaggle/working/Machine-Learning/{path}"

headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/vnd.github+json"
}

url = f"https://api.github.com/repos/{repo}/contents/{path}"

# Get existing file SHA
r = requests.get(url, headers=headers)
r.raise_for_status()

sha = r.json()["sha"]

# Encode corrected notebook
with open(local_path, "rb") as f:
    content = base64.b64encode(f.read()).decode("utf-8")

# Update GitHub
response = requests.put(
    url,
    headers=headers,
    json={
        "message": "Fix invalid English-French Transformer notebook",
        "content": content,
        "sha": sha
    }
)

print("Status:", response.status_code)
print("Message:", response.json().get("message"))
print("URL:", response.json().get("content", {}).get("html_url"))

Status: 200
Message: None
URL: https://github.com/NaveedAhmad-6522/Machine-Learning/blob/main/Natural-Language-Processing/English-French-Transformer/English-French-Transformer.ipynb
